<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-segmentation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab <b>Semantic Segmentation (BDD100k)</b> notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> traces training signals back to the exact samples producing them. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details.</div>

# Semantic Segmentation (BDD100k) with WeightsLab

Trains a small U-Net on a **BDD100k**-style dataset and instruments per-**sample** Dice (metric) and CrossEntropy (loss), so every mask's quality is traced back to the exact sample producing it.

### What you'll do
1. Install WeightsLab.
2. Download your dataset **zip from a Google Drive link** and unzip it (the training **code is inlined** below).
3. Set every knob in one **config** dict (like a `config.yaml`).
4. Wrap the dataloaders, model, optimizer, and per-sample Dice + CrossEntropy signals with the SDK.
5. Train while per-sample signals are captured, then (optionally) open the live **Weights Studio** UI.

> Data note: this notebook fetches your dataset from a **Drive share link** you provide (`DATASET_URL`); all training code (dataset, model, criterions, loops) is **inlined below** - no `main.py`, no full-repo clone. The zip should unpack to a BDD100k-style layout (`images/{train,val}/` + `labels/{train,val}/`), one semantic mask per image.

## Setup

On Colab, pick a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
# %pip install -q weightslab
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev7"
%pip install -q torchmetrics

### Fetch the dataset from Google Drive

Set **`DATASET_URL`** below to your Drive **share link** for the dataset **zip** (the file must be shared as *Anyone with the link*). The cell downloads it with [`gdown`](https://github.com/wkentaro/gdown), extracts it, and auto-detects the dataset root — the folder that contains an `images/` subdirectory (so it works even if your zip wraps everything in a top-level folder). `config["data_root"]` then points at that folder.

The zip should unpack to the BDD100k layout the dataset class expects:

```
<root>/
  images/{train,val}/   # .jpg / .png inputs
  labels/{train,val}/   # matching .png masks (same basename as the image)
```

In [ ]:
# ===== Dataset — local copy preferred (byte-identical to ws-seg-wow-moment-bdd), Drive as fallback =====
# This notebook is set up to reproduce the "ws-seg-wow-moment-bdd" experiment.
# If that experiment's exact dataset is available locally, use it directly so
# results are directly comparable. Otherwise (e.g. on Colab) fall back to the
# Drive zip below (a smaller toy dataset with the same layout/format).
LOCAL_DATA_ROOT = "/home/ubuntu/guillaume_plg/data/bdd/bdd8k"  # matches ws-seg-wow-moment-bdd/config.yaml's data_root

# Paste your Drive SHARE link for the dataset zip here (shared as "Anyone with the link").
DATASET_URL = "https://drive.google.com/file/d/1rXNsROVeneXp91VccOGIjtYWkkhWECBb/view?usp=sharing"

import os, sys, subprocess, zipfile


def _find_data_root(base):
    """Return the folder that holds an `images/` subdir (handles a wrapping top-level dir)."""
    if os.path.isdir(os.path.join(base, "images")):
        return base
    for d, subdirs, _ in os.walk(base):
        if "images" in subdirs:
            return d
    return base


if os.path.isdir(os.path.join(LOCAL_DATA_ROOT, "images")):
    print(f"Found the ws-seg-wow-moment-bdd dataset locally at {LOCAL_DATA_ROOT} — using it directly.")
    DATA_ROOT = LOCAL_DATA_ROOT
else:
    # gdown handles Drive downloads (incl. the large-file virus-scan confirmation).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown

    zip_path = "dataset.zip"
    extract_dir = "dataset"

    # Download (fuzzy=True lets gdown accept a full "file/d/<id>/view" share URL).
    if not os.path.exists(zip_path):
        gdown.download(url=DATASET_URL, output=zip_path, quiet=False, fuzzy=True)
    else:
        print(f"{zip_path} already present — skipping download.")

    # Extract.
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

    DATA_ROOT = _find_data_root(extract_dir)

print("Dataset root:", DATA_ROOT)
print("Contents:", sorted(os.listdir(DATA_ROOT)))
for _split in ("train", "val", "test"):
    _p = os.path.join(DATA_ROOT, "images", _split)
    if os.path.isdir(_p):
        print(f"  images/{_split}: {len(os.listdir(_p))} files")

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase. These are the external imports gathered from the example's `main.py` and `utils/` modules.

In [7]:
import os
os.environ['WEIGHTSLAB_LOG_LEVEL'] = 'DEBUG'
import time
import logging
import tempfile

import numpy as np
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

# Setup loggers (from main.py)
logging.basicConfig(level=logging.ERROR)
logging.getLogger("PIL").setLevel(logging.INFO)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

15/07/2026-17:28:56.434 INFO:root:setup_logging: WeightsLab logging initialized - Log file: /tmp/tmpa8wvavhc/weightslab_logs/weightslab_20260715_172856.log
15/07/2026-17:28:56.435 INFO:weightslab:<module>: WeightsLab package initialized - Log level: DEBUG, Log to file: True
15/07/2026-17:28:56.435 INFO:weightslab:<module>: 
╭─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                     │
│  $$\      $$\           $$\           $$\         $$\               $$\                 $$\         │
│  $$ | $\  $$ |          \__|          $$ |        $$ |              $$ |                $$ |        │
│  $$ |$$$\ $$ | $$$$$$\  $$\  $$$$$$\  $$$$$$$\  $$$$$$\    $$$$$$$\ $$ |       $$$$$$\  $$$$$$$\    │
│  $$ $$ $$\$$ |$$  __$$\ $$ |$$  __$$\ $$  __$$\ \_$$  _|  $$  _____|$$ |       \____$$\ $$  __$$\   │
│  $$$$  _$$$$ |$$$$$$$$ |$$ |$$ /  $$ |$$ |  $$ |

## 2. The dataset (inlined `utils/data.py`)

`BDD100kSegDataset` returns `(image, uid, mask, metadata)` per sample, where `mask` is a single `[H, W]` semantic mask (pixel value = class id). `seg_collate` stacks the batch into `images [B, C, H, W]` and `labels [B, H, W]` so signals are attributed per sample.

In [8]:
# ===== utils/data.py — SAMPLE-WISE semantic segmentation =====
import os
import torch
import numpy as np

from torchvision import transforms

from torch.utils.data import Dataset

from PIL import Image


# =============================================================================
# BDD100k segmentation dataset (sample-wise)
# =============================================================================
class BDD100kSegDataset(Dataset):
    """Returns (image, uid, mask, metadata) with ONE semantic mask per sample.

    Layout expected under `root`:
      images/{train,val}/   inputs (.jpg/.png)
      labels/{train,val}/   masks  (.png, same basename; pixel value = class id)

    `mask` is a single [H, W] int64 tensor of class ids (0..num_classes-1;
    `ignore_index` for void). No per-instance masks — signals are per sample.
    """

    def __init__(
        self,
        root,
        split="train",
        num_classes=6,
        class_names=None,
        ignore_index=255,
        image_size=256,
        max_samples=None
    ):
        super().__init__()
        self.root = root
        self.split = split
        self.num_classes = num_classes
        self.class_names = class_names
        self.ignore_index = ignore_index
        self.task_type = "segmentation"

        img_dir = os.path.join(root, "images", split)
        lbl_dir = os.path.join(root, "labels", split)

        image_files = [
            f
            for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        image_files = sorted(set(image_files))[:max_samples] if max_samples != None else sorted(set(image_files)) # Optionally limit number of samples for faster testing

        self.images = []
        self.masks = []
        for fname in image_files:
            img_path = os.path.join(img_dir, fname)
            base, _ = os.path.splitext(fname)
            lbl_name = base + ".png"
            lbl_path = os.path.join(lbl_dir, lbl_name)
            if os.path.exists(lbl_path):
                self.images.append(img_path)
                self.masks.append(lbl_path)

        if len(self.images) == 0:
            raise RuntimeError(f"No image/label pairs found in {img_dir} / {lbl_dir}")

        # Resize to a fixed SQUARE (image_size, image_size). A single int would
        # resize only the shorter edge and keep the aspect ratio, so non-square
        # inputs come out as e.g. [128, 227] and mismatch the prediction in the
        # loss. The model input is square (input_shape=(1, C, image_size, image_size)).
        self.image_transform = transforms.Compose(
            [
                transforms.Resize(
                    # size=(image_size, image_size),
                    size=image_size,  # single int = resize shorter edge, keep aspect ratio
                    interpolation=Image.BILINEAR
                ),
                transforms.ToTensor(),
            ]
        )
        self.mask_resize = transforms.Resize(
            # size=(image_size, image_size),
            size=image_size,
            interpolation=Image.NEAREST
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        """Returns (item, uid, target, metadata):
            - item: input image tensor [C, H, W]
            - uid: unique id for the sample (filename)
            - target: semantic mask tensor [H, W] (int64 class ids)
            - metadata: optional dict (original paths)
        """
        return self.get_items(idx, include_metadata=True, include_labels=True, include_images=True)

    def get_items(self, idx, include_metadata=False, include_labels=False, include_images=False):
        img_path = self.images[idx]
        mask_path = self.masks[idx]
        uid = os.path.basename(img_path)

        metadata = {
            'img_path': img_path,
            'mask_path': mask_path,
        } if include_metadata else None

        # Process image
        img_t = None
        if include_images:
            img = Image.open(img_path).convert("RGB")
            img_t = self.image_transform(img)

        # Process label: one semantic mask [H, W] per sample.
        mask_t = None
        if include_labels:
            mask = Image.open(mask_path)
            mask_r = self.mask_resize(mask)
            mask_np = np.array(mask_r, dtype=np.int64)
            # Mask must be a single-channel class-id map [H, W]. If the file is
            # RGB/RGBA (extra channel dim), keep the first channel so it matches
            # the model's [H, W] prediction.
            if mask_np.ndim > 2:
                mask_np = mask_np[..., 0]
            mask_t = torch.from_numpy(mask_np) # [H, W] int64

        return img_t, uid, mask_t, metadata


def seg_collate(batch):
    """Collate WL per-sample tuples for semantic segmentation.

    Each item is ``(img, uid, mask, metadata)`` where ``mask`` is a single
    [H, W] class-id tensor. Images and masks stack into batched tensors; uids
    and metadata stay as per-sample lists.

    Returns:
        images: FloatTensor [B, C, H, W]
        ids: list[str] of length B
        labels: LongTensor [B, H, W]
        metas: list[B] of metadata dicts
    """
    images = torch.stack([b[0] for b in batch], dim=0)
    ids = [b[1] for b in batch]
    labels = torch.stack([torch.as_tensor(b[2]) for b in batch], dim=0)
    metas = [b[3] if len(b) > 3 else None for b in batch]
    return images, ids, labels, metas

## 3. The model (inlined `utils/model.py`)

A compact U-Net (`SmallUNet`) with `task_type="segmentation"` so WeightsLab renders masks in Weights Studio.

In [9]:
# ===== utils/model.py =====
# =============================================================================
# Small UNet-ish segmentation model
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, image_size=256):
        super().__init__()
        # For WeightsLab
        self.task_type = "segmentation"
        self.num_classes = num_classes
        self.class_names = ["Background", "Ego Road", "Driveable Area", "Lane Line 1", "Lane Line 2", "Lane Line 3"]
        self.input_shape = (1, in_channels, image_size, image_size)

        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(64, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(64 + 64, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(32 + 32, 32)

        self.head = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        # Bottleneck
        b = self.bottleneck(p2)

        # Decoder
        u2 = self.up2(b)
        # Important: no `if` on shapes; always interpolate
        u2 = F.interpolate(u2, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        u1 = F.interpolate(u1, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        logits = self.head(d1) # [B, C, H, W]
        return logits

## 4. Losses & metrics (inlined `utils/criterions.py`)

Per-sample combined **CrossEntropy + Dice** (loss) and **Dice** (metric), ported to match the `ws-seg-wow-moment-bdd` experiment's `criterions.py::CrossEntropySegLoss` exactly (same formula, same per-class weights, same `ignore_index` handling — see that class's docstring below for a quirk in the reference project that's preserved here for faithful reproduction). Each scores the model's `[B, C, H, W]` output against the `[B, H, W]` mask and returns **one value per sample** (`[B]`), wrapped with `per_sample=True` so WeightsLab logs it per `sample_id`.

In [ ]:
# ===== utils/criterions.py — SAMPLE-WISE Dice (metric) + combined CE+Dice (loss) =====
# The loss below is ported to match ws-seg-wow-moment-bdd/criterions.py's
# CrossEntropySegLoss exactly (same formula, same per-class weighting, same
# ignore_index handling) so this notebook reproduces that experiment's loss.
import torch
import torch.nn as nn
import torch.nn.functional as F

_EPS = 1e-7  # matches PerSampleMultiClassDiceLoss's eps in ws-seg-wow-moment-bdd/criterions.py


class PerSampleMultiClassDiceLoss(nn.Module):
    """Per-sample, per-class Dice LOSS (1 - diceScore), averaged over classes -> [B].

    Ported verbatim (same formula/eps/masking) from
    ws-seg-wow-moment-bdd/criterions.py::PerSampleMultiClassDiceLoss.
    """

    def __init__(self, ignore_index=255, eps=_EPS, weights=None, ignore_classes=None):
        super().__init__()
        self.ignore_index = ignore_index
        self.eps = eps
        self.weights = weights
        self.ignore_classes = ignore_classes

    def forward(self, logits, targets):
        bs, num_classes, H, W = logits.shape
        probs = torch.softmax(logits, dim=1) # [B, C, H, W]
        valid = (targets != self.ignore_index) # [B, H, W]
        targets_safe = targets.clone()
        targets_safe[~valid] = 0
        one_hot = F.one_hot(targets_safe, num_classes=num_classes).permute(0, 3, 1, 2).float()

        valid_f = valid.unsqueeze(1).float() # [B, 1, H, W]
        probs = probs * valid_f
        one_hot = one_hot * valid_f

        intersection = (probs * one_hot).sum(dim=(2, 3)) # [B, C]
        cardinality = (probs + one_hot).sum(dim=(2, 3)) # [B, C]
        dice_per_class = 1.0 - (2.0 * intersection + self.eps) / (cardinality + self.eps) # [B, C] — a LOSS, not a score

        if self.ignore_classes:
            class_mask = torch.ones(num_classes, dtype=torch.bool, device=logits.device)
            class_mask[list(self.ignore_classes)] = False
            dice_per_class = dice_per_class[:, class_mask]
            weights = self.weights[class_mask] if self.weights is not None else None
        else:
            weights = self.weights

        if weights is not None:
            dice_per_class = dice_per_class * weights
        return dice_per_class.mean(dim=1) # [B]


class PerSampleCEDice(nn.Module):
    """Combined CrossEntropy + Dice loss -> [B] (differentiable, per-sample).

    Ported from ws-seg-wow-moment-bdd/criterions.py::CrossEntropySegLoss, with
    one fix applied on both sides: `PerSampleMultiClassDiceLoss` above already
    returns a dice LOSS (`1 - diceScore`), so it's used directly here. The
    reference project originally did `(1 - dice)` again on top of that, which
    algebraically reduces to the raw dice SCORE (i.e. a *better* dice score
    increased the loss) — that double-negation has been fixed in both this
    notebook and ws-seg-wow-moment-bdd/criterions.py.
    """

    def __init__(self, weights=None, ignore_index=255, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.register_buffer(
            "weights",
            torch.as_tensor(weights, dtype=torch.float32) if weights is not None else None,
        )
        self.ignore_index = ignore_index
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice = PerSampleMultiClassDiceLoss(ignore_index=ignore_index, weights=self.weights)

    def forward(self, outputs, labels):
        # outputs [B, C, H, W]; labels [B, H, W] (int64 class ids)
        ce = F.cross_entropy(
            outputs, labels.long(),
            weight=self.weights, ignore_index=self.ignore_index, reduction="none",
        ) # [B, H, W]
        # Mean over ALL pixels (ignored ones included as 0), matching the
        # source project's un-masked mean (not a valid-pixel-only average).
        ce = ce.flatten(1).mean(dim=1) # [B]

        dice_loss = self.dice(outputs, labels.long()) # [B] — already a LOSS (1 - diceScore)

        return self.bce_weight * ce + self.dice_weight * dice_loss


class PerSampleDice(nn.Module):
    """Per-sample mean foreground Dice -> [B] (metric, detached; for the WL dashboard)."""

    def forward(self, outputs, labels):
        probs = torch.softmax(outputs, dim=1) # [B, C, H, W]
        C = probs.shape[1]
        tgt = torch.where(labels < C, labels, torch.zeros_like(labels)).long() # void/ignore -> background
        onehot = F.one_hot(tgt, num_classes=C).permute(0, 3, 1, 2).float() # [B, C, H, W]
        inter = (probs * onehot).sum(dim=(2, 3))
        denom = probs.sum(dim=(2, 3)) + onehot.sum(dim=(2, 3))
        dice = (2.0 * inter + _EPS) / (denom + _EPS) # [B, C]
        fg = dice[:, 1:].mean(dim=1) if C > 1 else dice.mean(dim=1) # exclude background channel
        return fg.detach() # [B]

## 5. Configuration

Every tunable lives here in one dict - this is `config.yaml` inlined, with its comments. Wrapping it with `flag="hyperparameters"` lets Weights Studio read (and live-edit) these values while training. `data_root` points at the sample fetched above.

**Reproducing `ws-seg-wow-moment-bdd`:** `ignore_index`, `image_size`, `training_steps_to_do`, `eval_full_to_train_steps_ratio`, and the train `batch_size` below are set to match that experiment's `config.yaml` exactly, since these directly change training results (model architecture, optimizer, and dataset format were already identical).

In [ ]:
config = {
    # -- Global configuration ------------------------------------------------
    "device": "auto",                         # 'auto' -> resolved to `device` from the Imports cell
    "experiment_name": "seg_bdd_training",    # name shown in Weights Studio
    "training_steps_to_do": 10000,              # matches ws-seg-wow-moment-bdd/config.yaml (training_steps_to_do)
    "root_log_dir": None,                       # None -> a temp dir is created (history + dataframe land here)

    "checkpoint_manager": {
        "load_config": False,                   # don't reload a previous checkpoint's config, so edits here take effect
    },

    # Initially compute natural sorting values, e.g. average intensity (see the example README).
    "compute_natural_sort": True,

    # -- Experiment parameters ----------------------------------------------
    "eval_full_to_train_steps_ratio": 256,      # matches ws-seg-wow-moment-bdd (test every 256 steps)
    "experiment_dump_to_train_steps_ratio": 256,
    "write_export_ratio": 100,                  # export history + dataframe every N steps (main.py default)
    "tqdm_display": True,
    "is_training": False,                       # start training immediately or not

    # -- Serving (the notebook serves via gRPC + a bore tunnel; see the Serve cell) --
    "serving_grpc": True,
    "serving_cli": True,

    # -- Global dataframe storage -------------------------------------------
    "ledger_enable_h5_persistence": True,
    "ledger_enable_flushing_threads": True,
    "ledger_flush_max_rows": 100,
    "ledger_flush_interval": 60.0,

    # -- Data ----------------------------------------------------------------
    "num_classes": 6,
    "class_names": ["Background", "Ego Road", "Driveable Area", "Lane Line 1", "Lane Line 2", "Lane Line 3"],
    "ignore_index": 0,                          # matches ws-seg-wow-moment-bdd: excludes "Background" (class 0) from loss/metric
    "image_size": 720,                          # matches ws-seg-wow-moment-bdd (shortest edge resized to 720, aspect ratio kept)
    "data_root": DATA_ROOT,                     # local ws-seg-wow-moment-bdd dataset if found, else the Drive zip (see the data-fetch cell)
    "data": {
        "train_loader": {"batch_size": 8, "shuffle": True},   # matches ws-seg-wow-moment-bdd (train batch_size 8)
        "test_loader": {"batch_size": 2, "shuffle": False},
    },

    # -- Optimizer -----------------------------------------------------------
    "optimizer": {"lr": 1e-3},                  # Adam learning rate (matches ws-seg-wow-moment-bdd, which has no `optimizer:` key so falls back to this same 1e-3 default)
}

# Resolve the 'auto' device to the one picked in the Imports cell.
if config.get("device", "auto") == "auto":
    config["device"] = str(device)

# Register the config so Weights Studio can read (and live-edit) it while training.
wl.watch_or_edit(config, flag="hyperparameters", name=config["experiment_name"],
                 defaults=config, poll_interval=1.0)

# Logging directory: None -> a temp dir is created.
if not config.get("root_log_dir"):
    config["root_log_dir"] = tempfile.mkdtemp(prefix="weightslab_seg_")
os.makedirs(config["root_log_dir"], exist_ok=True)
log_dir = config["root_log_dir"]

# Schedule knobs used by the training loop.
eval_full_to_train_steps_ratio = config["eval_full_to_train_steps_ratio"]
write_export_ratio = config.get("write_export_ratio", 100)
tqdm_display = config.get("tqdm_display", True)
print("Experiment logs ->", log_dir)

## 6. Build data, model & optimizer

The heart of WeightsLab: each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave like the originals but report their state and per-sample signals. The loaders use `collate_fn=seg_collate` to stack the image and mask batches.

In [ ]:
# --- Hyperparameters read back after watch_or_edit ---
num_classes = int(config["num_classes"])
class_names = config["class_names"]
ignore_index = int(config["ignore_index"])
image_size = int(config["image_size"])

# --- Data (local ws-seg-wow-moment-bdd dataset, or the Drive zip fetched above) ---
data_root = config.get("data_root", DATA_ROOT)
if os.path.exists(data_root):
    print(f"Using data root: {data_root}")
else:
    raise FileNotFoundError(
        f"Data root not found: {data_root!r}. Run the data-fetch cell above first.")

train_cfg = config.get("data", {}).get("train_loader", {})
test_cfg = config.get("data", {}).get("test_loader", {})

# ws-seg-wow-moment-bdd's dataset uses images/{train,test}; the packaged Drive
# toy dataset uses images/{train,val} — detect whichever is actually present.
eval_split = "test" if os.path.isdir(os.path.join(data_root, "images", "test")) else "val"
print(f"Eval split: {eval_split!r}")

_train_dataset = BDD100kSegDataset(
    root=data_root,
    split="train",
    num_classes=num_classes,
    class_names=class_names,
    ignore_index=ignore_index,
    image_size=image_size,
    max_samples=train_cfg.get("max_samples", None),
)
_test_dataset = BDD100kSegDataset(
    root=data_root,
    split=eval_split,
    num_classes=num_classes,
    class_names=class_names,
    ignore_index=ignore_index,
    image_size=image_size,
    max_samples=test_cfg.get("max_samples", None),
)

train_loader = wl.watch_or_edit(
    _train_dataset, flag="data", loader_name="train_loader",
    batch_size=train_cfg.get("batch_size", 2), shuffle=train_cfg.get("shuffle", True),
    compute_hash=False, is_training=True,
    array_autoload_arrays=False, array_return_proxies=True, array_use_cache=True,
    preload_labels=False, collate_fn=seg_collate,
)
test_loader = wl.watch_or_edit(
    _test_dataset, flag="data", loader_name="test_loader",
    batch_size=test_cfg.get("batch_size", 2), shuffle=test_cfg.get("shuffle", False),
    compute_hash=False, is_training=False,
    array_autoload_arrays=False, array_return_proxies=True, array_use_cache=True,
    preload_labels=True, collate_fn=seg_collate,
)

_model = SmallUNet(in_channels=3, num_classes=num_classes, image_size=image_size).to(device)
model = wl.watch_or_edit(_model, flag="model", device=device, compute_dependencies=True)

lr = config.get("optimizer", {}).get("lr", 1e-3)
_optimizer = optim.Adam(model.parameters(), lr=lr)
optimizer = wl.watch_or_edit(_optimizer, flag="optimizer")

## 7. Train & eval steps (+ tracked signals)

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `_run_signals` computes and logs the per-sample **Dice** (metric) and **CrossEntropy** (loss); `wl.save_signals(...)` records custom per-sample diagnostics (predicted vs present classes). Class weights are computed from the training split, then the tracked signals are built.

In [ ]:
# ===== train / eval step functions (sample-wise) =====
from torchmetrics import JaccardIndex


def _run_signals(sig, outputs, labels, ids, preds, return_metric=False):
    """Compute + log the per-sample combined CE+Dice (loss) and Dice (metric)."""
    loss_sample = sig["loss_sample"](outputs, labels, batch_ids=ids, preds=preds)
    dice_sample = sig["dice_sample"](outputs, labels, batch_ids=ids)
    if return_metric:
        return loss_sample, dice_sample
    return loss_sample


def _user_custom_signals(preds, labels):
    """Per-sample diagnostics: which classes are predicted vs present in the mask."""
    B = preds.shape[0]
    return {
        "preds_classes_per_sample": [preds[i].unique() for i in range(B)],
        "target_classes_per_sample": [labels[i].unique() for i in range(B)],
        "tp_classes_per_sample": [
            torch.tensor([c for c in labels[i].unique() if c in preds[i].unique()]) for i in range(B)
        ],
        "fp_classes_per_sample": [
            torch.tensor([c for c in preds[i].unique() if c not in labels[i].unique()]) for i in range(B)
        ],
        "fn_classes_per_sample": [
            torch.tensor([c for c in labels[i].unique() if c not in preds[i].unique()]) for i in range(B)
        ],
    }


def train(loader, model, optimizer, sig, metric, device):
    """Single training step. loader yields (inputs, ids, labels, metadata);
    `labels` is a [B, H, W] semantic mask (see seg_collate)."""
    with guard_training_context:
        (inputs, ids, labels, _) = next(loader)
        inputs = inputs.to(device)
        labels = labels.to(device) # [B, H, W]

        optimizer.zero_grad()
        outputs = model(inputs) # [B, C, H, W]
        preds = outputs.argmax(dim=1) # [B, H, W]

        # Per-sample combined CE+Dice (loss) + Dice (metric), tracked & saved per sample.
        loss_per_sample = _run_signals(sig, outputs, labels, ids, preds=preds) # [B]
        loss = loss_per_sample.mean()

        loss.backward()
        optimizer.step()

        # Per-sample class diagnostics for the UI (predicted vs present classes).
        wl.save_signals(_user_custom_signals(preds, labels), ids)

    # Batch IoU, matching ws-seg-wow-moment-bdd's train() (reset every step).
    metric.update(preds, labels.detach())
    train_iou = metric.compute().detach().cpu().item() * 100.0
    metric.reset()

    return float(loss.detach().cpu().item()), train_iou


def test(loader, model, sig, metric, device, test_loader_len):
    """Full evaluation pass over the eval loader."""
    losses = torch.tensor([0.]).to(device)
    dices = torch.tensor([0.]).to(device)
    metric.reset()
    with guard_testing_context, torch.no_grad():
        for inputs, ids, labels, _ in loader:
            inputs = inputs.to(device)
            labels = labels.to(device) # [B, H, W]

            outputs = model(inputs)
            preds = outputs.argmax(dim=1) # [B, H, W]

            loss_per_sample, dice_sample = _run_signals(sig, outputs, labels, ids, preds=preds, return_metric=True)
            losses += torch.mean(loss_per_sample) # accumulate batch mean
            dices += torch.mean(dice_sample)
            metric.update(preds, labels)

            wl.save_signals(_user_custom_signals(preds, labels), ids)

    loss = float((losses / test_loader_len).detach().cpu().item())
    dice = float((dices / test_loader_len).detach().cpu().item())
    iou = metric.compute().detach().cpu().item() * 100.0
    return loss, dice * 100.0, iou # Dice/IoU as percentages


# ===== tracked per-sample Dice (metric) + combined CE+Dice (loss) signals =====
# Both are wrapped with per_sample=True so WeightsLab logs one value per
# sample_id. No per-instance signals.
def _make_seg_signals(split: str, weights=None, ignore_index: int = 255) -> dict:
    return {
        "dice_sample": wl.watch_or_edit(
            PerSampleDice(), flag="metric",
            name=f"{split}_dice/sample", per_sample=True, log=True,
        ),
        "loss_sample": wl.watch_or_edit(
            PerSampleCEDice(weights=weights, ignore_index=ignore_index), flag="loss",
            name=f"{split}_CE", per_sample=True, log=True,
        ),
    }


def compute_class_weights(dataset, num_classes, ignore_index=255, max_samples=100):
    print("\n" + "=" * 60, flush=True)
    print(f"Computing class weights for {num_classes} classes (max {max_samples} samples)...", flush=True)
    class_counts = np.zeros(num_classes, dtype=np.float64)
    num_samples = min(len(dataset), max_samples)

    for idx in tqdm.tqdm(range(num_samples), desc=" Analyzing Distribution"):
        _, _, label, _ = dataset.get_items(idx, include_labels=True) # semantic mask [H, W]
        label_np = label.numpy() if hasattr(label, 'numpy') else np.array(label)
        for c in range(num_classes):
            class_counts[c] += (label_np == c).sum()

    class_counts = np.maximum(class_counts, 1) # Avoid div by zero
    total_pixels = class_counts.sum()
    class_weights = total_pixels / (num_classes * class_counts)
    class_weights = class_weights / class_weights.mean() # Normalize

    print("\nClass distribution and weights:", flush=True)
    for c in range(num_classes):
        pct = (class_counts[c] / total_pixels) * 100
        print(f"Class {c}: {pct:6.2f}% -> weight: {class_weights[c]:.3f}", flush=True)
    print("=" * 60 + "\n", flush=True)
    return torch.FloatTensor(class_weights).to(device)

weights = compute_class_weights(_train_dataset, num_classes, max_samples=100)

# Matches ws-seg-wow-moment-bdd: the train criterion is class-weighted, the
# test criterion is NOT (its CrossEntropySegLoss gets no `weights=` argument there).
train_sig = _make_seg_signals("train", weights=weights, ignore_index=ignore_index)
test_sig = _make_seg_signals("test", weights=None, ignore_index=ignore_index)

# IoU (macro, ignore_index=0), matching ws-seg-wow-moment-bdd's train_iou/test_iou.
train_metric = wl.watch_or_edit(
    JaccardIndex(task="multiclass", num_classes=num_classes, ignore_index=ignore_index, average="macro").to(device),
    flag="metric", name="train_iou", per_sample=False, log=True,
)
test_metric = wl.watch_or_edit(
    JaccardIndex(task="multiclass", num_classes=num_classes, ignore_index=ignore_index, average="macro").to(device),
    flag="metric", name="test_iou", per_sample=False, log=True,
)

## 8. Serve and train

`wl.serve(serving_grpc=True, serving_bore=True)` starts the background gRPC server and opens a public `bore.pub:<port>` tunnel (printed below) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run a finite loop, periodically evaluating and exporting signals.

Leave this cell **running** while you watch it stream in the UI.

In [ ]:
wl.serve(serving_grpc=config.get("serving_grpc", True), serving_bore=True)
os.environ['WL_DEBUG'] = '1'
wl.start_training(timeout=3)

# training_steps_to_do may be None (open-ended); cap it here so the notebook run is finite.
max_steps = config["training_steps_to_do"] or 500
train_range = tqdm.tqdm(range(max_steps), desc="Training") if tqdm_display else range(max_steps)
test_loss, test_dice, test_iou = None, None, None
start_time = time.time()
for train_step in train_range:
    age = model.get_age() if hasattr(model, "get_age") else train_step

    # Train one step
    train_loss, train_iou = train(train_loader, model, optimizer, train_sig, train_metric, device)

    # Full evaluation pass
    if age == 0 or age % eval_full_to_train_steps_ratio == 0:
        test_loader_len = len(test_loader)
        test_loader_it = tqdm.tqdm(test_loader, desc="Evaluating", leave=False) if tqdm_display else test_loader
        test_loss, test_dice, test_iou = test(test_loader_it, model, test_sig, test_metric, device, test_loader_len)

    # Periodic history + dataframe export (JSON/CSV snapshots to root_log_dir)
    if age > 0 and age % write_export_ratio == 0:
        wl.write_history()
        wl.write_dataframe()

    if tqdm_display:
        train_range.set_description("Step")
        train_range.set_postfix(
            train_loss=f"{train_loss:.4f}",
            train_iou=f"{train_iou:.2f}%",
            test_loss=f"{test_loss:.4f}" if test_loss is not None else "N/A",
            test_dice=f"{test_dice:.2f}%" if test_dice is not None else "N/A",
            test_iou=f"{test_iou:.2f}%" if test_iou is not None else "N/A",
        )

print("\n" + "=" * 60)
print(f" Training completed in {time.time() - start_time:.2f} seconds")
print(f" Logs saved to: {log_dir}")
print("=" * 60)

# Final export of signal history and data grid to root_log_dir
wl.write_history()
wl.write_dataframe()

## See it live in Weights Studio

**Colab has no Docker daemon**, so run Studio on your own machine and point it at this notebook's
backend using the `bore.pub:<port>` printed in the Expose section:

```bash
pip install weightslab
weightslab ui launch                       # Terminal 1 - opens http://localhost:5173
weightslab tunnel bore.pub:12345           # Terminal 2 - the host:port from the Expose section
```

Then open **http://localhost:5173**. Prefer local-only? Run this example directly on your machine
(`weightslab start example` selects a bundled example) and launch the UI next to it - no tunnel.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>